In [ ]:
#!pip install tenacity==9.0.0
#!pip install langchain==0.3.12
#!pip install langchain-openai==0.2.12
#!pip install langchain_community==0.3.12
#!pip install langgraph==0.2.59
#!pip install pysqlite3-binary==0.5.4
#!pip install langchain_chroma==0.1.4
#!pip install pandas==2.2.3
#!pip install -U pypdf
#!pip install nbformat==5.10.4
#!pip install pysqlite3
#!pip install "posthog>=2.4.0,<6.0.0" #Fix chroma error
#!pip install fuzzywuzzy #in case a typo
#!pip install python-Levenshtein #for fuzzywuzzy packg

  Using cached langchain_openai-0.2.12-py3-none-any.whl.metadata (2.7 kB)
  Using cached openai-1.109.1-py3-none-any.whl.metadata (29 kB)
Using cached langchain_openai-0.2.12-py3-none-any.whl (50 kB)
Using cached openai-1.109.1-py3-none-any.whl (948 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.3.11
    Uninstalling langgraph-sdk-0.3.11:
      Successfully uninstalled langgraph-sdk-0.3.11
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0.1
    Uninstalling langgraph-checkpoint-4.0.1:
      Successfully uni

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 94.4 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: posthog
    Found existing installation: posthog 7.9.12
    Uninstalling posthog-7.9.12:
      Successfully uninstalled posthog-7.9.12


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from langchain_openai import ChatOpenAI
from google.colab import userdata
from langchain_openai import OpenAIEmbeddings # We are using OpenAIEmbedding instead of Azure
import os


# API info. Replace with your own keys
os.environ["OPENAI_API_KEY"] = userdata.get("OpenAiAPI")

# Setup the LLM
model = ChatOpenAI(
    model="gpt-5.1",
    temperature=0.7
)

#Setup the Embedding
embedding = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=userdata.get("OpenAiAPI"),
)


In [ ]:
import pandas as pd
from langchain_core.tools import tool
from fuzzywuzzy import fuzz

#Load the pc part pricing CSV into a Pandas dataframe.
product_pricing_df = pd.read_csv("/content/drive/MyDrive/Group 3 - Agentic AI in Customer Service and Sales/data/pc_parts_pricing.csv")
print(product_pricing_df)

@tool
def get_pc_part_price(parts_name:str) -> str :
    """
    This function returns the price of a pc parts, given its name as input.
    It performs a substring match between the input name and the pc part name.
    If a match is found, it returns the pricxe of that pc part.
    If there is NO match found, it returns -1
    """

    best_match_score = -1
    best_match_index = -1

    for index, row in product_pricing_df.iterrows():
        ratio = fuzz.ratio(parts_name.lower(), row["Name"].lower())
        if ratio > best_match_score:
            best_match_score = ratio
            best_match_index = index

    if best_match_score >= 60:
        return f"{product_pricing_df['Name'].iloc[best_match_index]}: ${product_pricing_df['Price'].iloc[best_match_index]}"
    else:
        return "Not Found in the System"

#Test the tool. Before running the test, comment the @tool annotation
print(get_pc_part_price.invoke({"parts_name": "NVIDIA RTX 4070"}))

                                 Name  Price  ShippingDays
0                 WD Black SN850X 1TB    312             7
1                  NVIDIA RTX 4070 Ti   1421             3
2           Seagate Barracuda 2TB HDD   1047             5
3             Cooler Master Hyper 212    994             1
4               Seasonic Focus GX-850    677             5
..                                ...    ...           ...
495                      NZXT H7 Flow    355             7
496               Crucial P3 1TB NVMe    264             6
497  G.Skill Trident Z 32GB DDR5 6400    226             5
498                 AMD Ryzen 9 5900X    105             7
499                     Noctua NH-D15    332             1

[500 rows x 3 columns]
NVIDIA RTX 4070: $956


In [ ]:
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

from langchain_core.tools import create_retriever_tool
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load, chunk and index the contents of the product featuers document.
loader=PyPDFLoader("/content/drive/MyDrive/Group 3 - Agentic AI in Customer Service and Sales/data/pc_parts_descriptions_full.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=256)
splits = text_splitter.split_documents(docs)

#Create a vector store with Chroma
prod_feature_store = Chroma.from_documents(
    documents=splits,
    embedding=embedding
)

get_product_features = create_retriever_tool(
    prod_feature_store.as_retriever(search_kwargs={"k": 4}),
    name="Get_Product_Features",
    description="""
    This store is PC parts selling store. contains details about all of the PC parts. It lists the available PC part name
    and their features including design and advantages, detailed specs of the part.
    """
)

#Test the product feature store
print(prod_feature_store.as_retriever().invoke("Tell me about the NDVIA 4080") )

[Document(metadata={'page': 23, 'source': '/content/drive/MyDrive/Group 3 - Agentic AI in Customer Service and Sales/data/pc_parts_descriptions_full.pdf'}, page_content='performance and usability for a variety of computing needs.\nNVIDIA RTX 4080\nThe NVIDIA RTX 4080 is a reliable PC component designed to enhance overall system\nperformance and usability for a variety of computing needs.\nCorsair Vengeance 32GB DDR5 6000\nThe Corsair Vengeance 32GB DDR5 6000 is a reliable PC component designed to enhance overall\nsystem performance and usability for a variety of computing needs.\nAMD Ryzen 9 5900X\nThe AMD Ryzen 9 5900X is a reliable PC component designed to enhance overall system\nperformance and usability for a variety of computing needs.\nSamsung 980 PRO 2TB NVMe\nThe Samsung 980 PRO 2TB NVMe is a reliable PC component designed to enhance overall\nsystem performance and usability for a variety of computing needs.\nNVIDIA RTX 3060\nThe NVIDIA RTX 3060 is a reliable PC component desig

## Setup a Product QnA chatbot

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage

#Create a System prompt to provide a persona to the chatbot
system_prompt = SystemMessage("""
    You are professional chatbot that answers questions about computer and computer part sold by your company.
    To answer questions about computer and computer parts question, you will ONLY use the available tools and NOT your own memory.
    You will handle small talk and greetings by producing professional responses.
    """
)

#Create a list of tools available
tools = [get_pc_part_price, get_product_features]

#Create memory across questions in a conversation (conversation memory)
checkpointer=MemorySaver()

#Create a Product QnA Agent. This is actual a graph in langGraph
product_QnA_agent=create_react_agent(
                                model=model, #LLM to use
                                tools=tools, #List of tools to use
                                state_modifier=system_prompt, #The system prompt
                                debug=False, #Debugging turned on if needed
                                checkpointer=checkpointer #For conversation memory
)

In [ ]:
#Setup chatbot
import uuid
#To maintain memory, each request should be in the context of a thread.
#Each user conversation will use a separate thread ID
config = {"configurable": {"thread_id": uuid.uuid4()}}

#Test the agent with an input
inputs = {"messages":[
                HumanMessage("What are the features and pricing of NVIDIA RTX 4070 ti?")
            ]}

#Use streaming to print responses as the agent  does the work.
#This is an alternate way to stream agent responses without waiting for the agent to finish
for stream in product_QnA_agent.stream(inputs, config, stream_mode="values"):
    message=stream["messages"][-1]
    if isinstance(message, tuple):
        print(message)
    else:
        message.pretty_print()

================================ Human Message =================================

What are the features and pricing of NVIDIA RTX 4070 ti?
================================== Ai Message ==================================
Tool Calls:
  Get_Product_Features (call_UJ5h6GK4LPHmzfk4CzORTaa9)
 Call ID: call_UJ5h6GK4LPHmzfk4CzORTaa9
  Args:
    query: NVIDIA RTX 4070 ti
  get_pc_part_price (call_q0bF47NoC7bwnFy5uB7G7DYZ)
 Call ID: call_q0bF47NoC7bwnFy5uB7G7DYZ
  Args:
    parts_name: NVIDIA RTX 4070 ti
================================= Tool Message =================================
Name: get_pc_part_price

NVIDIA RTX 4070 Ti: $1421
================================== Ai Message ==================================

The **NVIDIA RTX 4070 Ti** is a reliable PC component designed to enhance overall system performance and usability for a variety of computing needs. It is suitable for operating systems, applications, and large file storage.

The price of the NVIDIA RTX 4070 Ti is **$1421**.


## Test the Product QnA Chatbot

In [ ]:
import uuid
#Send a sequence of messages to chatbot and get its response
#This simulates the conversation between the user and the Agentic chatbot
user_inputs = [
    "Hello",
    "I am looking to buy parts for high end gaming pc",
    "Give me a list of available parts names",
    "Tell me about the features of the parts that you listed",
    "How much does it cost?",
    "Thanks for the help"
]

#Create a new thread
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

for input in user_inputs:
    print(f"----------------------------------------\nUSER : {input}")
    #Format the user message
    user_message = {"messages":[HumanMessage(input)]}
    #Get response from the agent
    ai_response = product_QnA_agent.invoke(user_message,config=config)
    #Print the response
    print(f"AGENT : {ai_response['messages'][-1].content}")


----------------------------------------
USER : Hello
AGENT : Hello! How can I assist you today with your computer or computer parts inquiries?
----------------------------------------
USER : I am looking to buy parts for high end  gaming pc
AGENT : Great choice! Building a high-end gaming PC can provide an exceptional gaming experience. Could you please specify which parts you're interested in or if you need recommendations for components like the CPU, GPU, RAM, motherboard, etc.? Let me know how I can assist you further!
----------------------------------------
USER : Give me a list of available parts names
AGENT : Here is a list of available high-end PC parts you can consider for your gaming setup:

1. **Lian Li PC-O11 Dynamic** - A stylish and spacious PC case.
2. **ASUS TUF Gaming B650-PLUS** - A robust motherboard for gaming.
3. **Intel Core i7-12700K** - A powerful CPU for enhanced performance.
4. **Seagate Barracuda 2TB HDD** - Reliable storage with fast data access.
5. **AMD R

In [ ]:
import uuid
#conversation memory by user
def execute_prompt(user, config, prompt):
    inputs = {"messages":[("user",prompt)]}
    ai_response = product_QnA_agent.invoke(inputs,config=config)
    print(f"\n{user}: {ai_response['messages'][-1].content}")

#Create different session threads for 2 users
config_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}
config_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

#Test both threads
#execute_prompt("USER 1", config_1, "Tell me about the features of RTX 4070 Ti")
#execute_prompt("USER 2", config_2, "Tell me about the features of Noctua NH-D15")
#execute_prompt("USER 1", config_1, "What is its price ?")
#execute_prompt("USER 2", config_2, "What is its price ?")


USER 1: The NVIDIA RTX 4070 Ti is a reliable PC component designed to enhance overall system performance and usability. It is suitable for a variety of computing needs, including operating systems, applications, and large file storage.

USER 2: The Noctua NH-D15 is a reliable PC component designed to enhance overall system performance and usability for a variety of computing needs.

USER 1: The price of the NVIDIA RTX 4070 Ti is $1421.

USER 2: The price of the Noctua NH-D15 is $1282.
